# Delle Monache, Petrella & Venditti (2021) — Monte Carlo Simulation

Score-driven state-space model of the price-dividend ratio and long-run stock returns.

**Reference:** Delle Monache, Petrella & Venditti (2020/2021), *Price Dividend Ratio and Long-Run Stock Returns: A Score-Driven State Space Model*, ECB WP 2369 / JBES 39(4), 1054–1065.

---

### Model dynamics (quarterly)

**Permanent components** (score-driven martingales):
$$\bar{g}_t = \bar{g}_{t-1} + \sigma_{\bar{g}}\,\eta_{\bar{g}} \qquad \bar{\mu}_t = \bar{\mu}_{t-1} + \sigma_{\bar{\mu}}\,\eta_{\bar{\mu}}$$

**Transitory components**:
$$\tilde{g}_t = \tilde{g}_{t-1} + \varepsilon_{g,t} \qquad (\text{random walk})$$
$$\tilde{\mu}_t = \rho_\mu\,\tilde{\mu}_{t-1} + \varepsilon_{\mu,t} \qquad (\text{AR(1)})$$

**Dividend growth** (expected + unexpected):
$$\Delta d_t = (\bar{g}_{t-1} + \tilde{g}_{t-1}) + \varepsilon_{d,t} + \varepsilon_{g,t}$$

**Log price-dividend ratio** (Campbell-Shiller present value):
$$pd_t = \kappa + \frac{\bar{g}_t + \tilde{g}_t - \bar{\mu}_t}{1-\rho} - \frac{\tilde{\mu}_t}{1-\rho\rho_\mu}$$

**Log return** (Campbell-Shiller identity):
$$r_t = \Delta d_t + \log(1 + e^{pd_t}) - pd_{t-1}$$

Structural shocks $(\varepsilon_d, \varepsilon_g, \varepsilon_\mu)$ are correlated with variances $(\sigma_d^2, \sigma_g^2, \sigma_\mu^2)$ and correlations $\rho_{d\mu}, \rho_{g\mu}$.

## Setup

In [ ]:
include(joinpath(@__DIR__, "DelleMonacheModel.jl"))
using .DelleMonacheModel
using Random, Statistics, Printf

## Parameters

Parameters calibrated to quarterly US equity data (CRSP, 1926–2018), consistent with the ML estimates from Table 2 of the paper.

In [ ]:
p = default_params()
κ = calibrate_kappa(p)

println("Model parameters (quarterly):")
println("  Campbell-Shiller ρ  = ", p.ρ)
println("  Transitory ER ρ_μ   = ", p.ρ_μ,  "  (annual: ", round(p.ρ_μ^4, digits=4), ")")
println("  Permanent growth ḡ₀ = ", p.ḡ₀,  "  (", round(p.ḡ₀*400, digits=2), "% pa)")
println("  Permanent ER μ̄₀     = ", p.μ̄₀,  "  (", round(p.μ̄₀*400, digits=2), "% pa)")
println("  σ_d = ", p.σ_d, "  σ_g = ", p.σ_g, "  σ_μ = ", p.σ_μ)
println("  ρ_dμ = ", p.ρ_dμ, "  ρ_gμ = ", p.ρ_gμ)
println("  κ (calibrated)      = ", round(κ, digits=4))
println("  Steady-state pd_ss  = ", p.pd_ss, "  (P/D ≈ ", round(exp(p.pd_ss), digits=1), ")")

## Simulation Configuration

In [ ]:
N_PATHS      = 10_000
T_QUARTERS   = 100          # 25 years
BASE_SEED    = 20212021     # Delle Monache JBES 2021
HORIZONS_YRS = [1, 5, 10, 25]
HORIZONS_Q   = HORIZONS_YRS .* 4

println("N_PATHS    = ", N_PATHS)
println("T_QUARTERS = ", T_QUARTERS, "  (", T_QUARTERS÷4, " years)")
println("Horizons   = ", HORIZONS_YRS, " years")

## Data Structures

In [ ]:
"""
    HorizonSnapshot

Cross-sectional values across all MC paths at a single horizon.
Field `c` = transitory growth g̃_t (cycle-like component).
"""
struct HorizonSnapshot
    horizon_years::Int
    d::Vector{Float64}          # log dividends
    g::Vector{Float64}          # total expected growth (quarterly)
    c::Vector{Float64}          # transitory growth g̃_t
    μ::Vector{Float64}          # total expected return (quarterly)
    pd::Vector{Float64}         # log price-dividend ratio
    cum_log_r::Vector{Float64}  # cumulative log return from t=0
    ann_log_r::Vector{Float64}  # annualised log return
end

struct MCResults
    params::ModelParams
    n_paths::Int
    t_quarters::Int
    snapshots::Dict{Int, HorizonSnapshot}
    sample_paths::Vector{SimulationResult}
end

## Monte Carlo Simulation

In [ ]:
function run_monte_carlo(; n_paths=N_PATHS, t_quarters=T_QUARTERS,
                           horizons_q=HORIZONS_Q, horizons_yrs=HORIZONS_YRS)

    valid_mask   = horizons_q .<= t_quarters
    valid_h_q    = horizons_q[valid_mask]
    valid_h_y    = horizons_yrs[valid_mask]

    collectors = Dict{Int, HorizonSnapshot}()
    for (hq, hy) in zip(valid_h_q, valid_h_y)
        collectors[hy] = HorizonSnapshot(
            hy,
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths)
        )
    end

    sample_indices = [1, div(n_paths, 2), n_paths]
    sample_paths   = Vector{SimulationResult}(undef, 3)

    print("  Simulating $n_paths paths × $t_quarters quarters...")
    t_start = time()

    for i in 1:n_paths
        sim = simulate_path(p, t_quarters; seed=BASE_SEED + i)

        for (hq, hy) in zip(valid_h_q, valid_h_y)
            vals = extract_at_horizon(sim, hq)
            snap = collectors[hy]
            snap.d[i]         = vals.d
            snap.g[i]         = vals.g
            snap.c[i]         = vals.g̃
            snap.μ[i]         = vals.μ
            snap.pd[i]        = vals.pd
            snap.cum_log_r[i] = vals.cum_log_r
            snap.ann_log_r[i] = annualised_cumulative_return(vals.cum_log_r, Float64(hy))
        end

        idx_in_samples = findfirst(==(i), sample_indices)
        if !isnothing(idx_in_samples)
            sample_paths[idx_in_samples] = sim
        end
    end

    elapsed = round(time() - t_start, digits=2)
    println(" done in $(elapsed)s")

    return MCResults(p, n_paths, t_quarters, collectors, sample_paths)
end

In [ ]:
results = run_monte_carlo()

## Save Results to CSV

In [ ]:
function save_results(results::MCResults, output_dir::String)
    mkpath(output_dir)

    for (hy, snap) in results.snapshots
        fname = joinpath(output_dir, "horizon_$(hy)yr.csv")
        open(fname, "w") do io
            println(io, "d,g,c,mu,pd,cum_log_r,ann_log_r")
            for i in 1:results.n_paths
                println(io,
                    "$(snap.d[i]),$(snap.g[i]),$(snap.c[i]),$(snap.μ[i])," *
                    "$(snap.pd[i]),$(snap.cum_log_r[i]),$(snap.ann_log_r[i])")
            end
        end
        println("  Saved: $fname")
    end

    for (k, sim) in enumerate(results.sample_paths)
        fname = joinpath(output_dir, "sample_path_$(k).csv")
        n     = sim.T + 1
        open(fname, "w") do io
            println(io, "quarter,d,tau,g,c,mu,pd,cum_log_r")
            for t in 1:n
                q = t - 1
                println(io,
                    "$q,$(sim.d[t]),$(sim.ḡ[t]),$(sim.g[t]),$(sim.g̃[t])," *
                    "$(sim.μ[t]),$(sim.pd[t]),$(sim.cum_log_r[t])")
            end
        end
        println("  Saved: $fname")
    end

    println("  All results saved to: $output_dir")
end

output_dir = joinpath(@__DIR__, "results")
save_results(results, output_dir)

## Summary Statistics

In [ ]:
println("═══════════════════════════════════════════════════════")
println("  Summary Statistics")
println("═══════════════════════════════════════════════════════")

quantiles_list = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

for hy in sort(collect(keys(results.snapshots)))
    snap = results.snapshots[hy]
    println("\n── Horizon: $(hy) years ──")

    println("  Log Dividends (d):")
    println("    Mean: $(round(mean(snap.d), digits=4))   Std: $(round(std(snap.d), digits=4))")

    println("  Total Expected Growth (annualised %):")
    println("    Mean: $(round(mean(snap.g)*400, digits=2))%   Std: $(round(std(snap.g)*400, digits=2))%")

    println("  Transitory Growth g̃_t (annualised %):")
    println("    Mean: $(round(mean(snap.c)*400, digits=2))%   Std: $(round(std(snap.c)*400, digits=2))%")

    println("  Log Price-Dividend Ratio (pd):")
    println("    Mean: $(round(mean(snap.pd), digits=4))   Std: $(round(std(snap.pd), digits=4))")

    println("  Annualised Log Return:")
    q_vals = quantile(snap.ann_log_r, quantiles_list)
    println("    Mean: $(round(mean(snap.ann_log_r)*100, digits=2))%   Std: $(round(std(snap.ann_log_r)*100, digits=2))%")
    for (q, v) in zip(quantiles_list, q_vals)
        @printf("    Q%-3s  %+.2f%%\n", "$(round(Int, q*100))%", v*100)
    end
end

## Scenario Analysis

Run the full quantile and distribution analysis using `analyse.jl`.

In [ ]:
include(joinpath(@__DIR__, "analyse.jl"))